In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import sys
import os

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.MSK_Model import MusculoskeletalSimulation, ControlMode
from src.visualizer import MusculoskeletalVisualizer
from src.IKParams import IK_Params, IK_Target, IK_Target_Mode
from src.IK_Solver import IK_Solver, IK_Algorithm
# import matplotlib.pyplot as plt
import mujoco
import time
import itertools
from scipy.spatial.transform import Rotation as R
import src.utilities as ut

import mediapy as media

In [46]:
# Initialize simulation with MyoSuite-style model
sim = MusculoskeletalSimulation('../models/myo_sim/arm/myoarm_IMU.xml')
mujoco.mj_forward(sim.model, sim.data)
# Set muscle control mode
muscle_params = {
    'activation_dynamics': False,
    'tau_activation': 0.015,
    'tau_deactivation': 0.060
}
sim.set_control_mode(ControlMode.MUSCLE, muscle_params)
viz = MusculoskeletalVisualizer(sim, azimuth=90, elevation=0, distance=1,lookat=[0, -0.5, 1.2])


iksol = IK_Solver(MSK_model=sim,viz=viz)
#iksol.IK_method = IK_Algorithm.Levenburg_Marquadt
iksol.IK_method = IK_Algorithm.Newton_Raphson
iksol.ik_prm.target_type = IK_Target_Mode.rot_only.value

mujoco.mj_forward(sim.model, sim.data)

sim.integrate = True
duration = 5
with mujoco.viewer.launch_passive(
            sim.model, 
            sim.data,
            show_left_ui=False,show_right_ui=False
        ) as viz.viewer:
        viz._viewer_settings()
            
        # viz.viewer.user_scn.flags[mujoco.mjtRndFlag.mjRND_WIREFRAME] = 1
        # viz.viewer.sync()

        start_time = time.time()
        sim_start_time = viz.sim.data.time
        last_log_time = 0
        
        # for site_pos in IK_target.site_targets.values():
        #     ut.draw_frame(viz.viewer,
        #                   site_pos[:3],
        #                   R.from_euler('xyz',site_pos[3:]).as_rotvec(),
        #                   AxisLen=0.2
        #                 )
        viz.render()
        # print(viz.viewer.opt.flags)
        tg = sim.get_site_pos('IMU_Hum')[3:] 
        IK_target = IK_Target()
        while viz.viewer.is_running() and (viz.sim.data.time - sim_start_time) < duration:
            # Get control input
            current_time = viz.sim.data.time
            current_pose = viz.sim.data.qpos.copy() 

            #sim.data.qpos = sim.model.jnt_range[:,0] + np.random.rand(len(sim.data.qpos))* (sim.model.jnt_range[:,1]-sim.model.jnt_range[:,0])
           
            
            #print(tg)
            IK_target.site_targets = {'IMU_Hum':tg,
                                    }
            iksol.set_target(IK_target)
            iksol.cal_error()
            iksol.solve()
            #print(f"status: {iksol.results.status}  iter: {iksol.results.iter}")
            #print(f"status: {viz.sim.data.time - sim_start_time} ")
            #print(sim.get_site_pos('IMU_Hum')[3:])
            
            #sim.data.qpos = iksol.results.qpos
            #mujoco.mj_forward(sim.model, sim.data)
            #sim.step(np.zeros(sim.model.nu))

            tg -= np.array([0.1*np.pi/180, 0.1*np.pi/180, 0])
            #tg -= np.array([0, 0, 0])
            
            viz.viewer.user_scn.ngeom = 0
            #viz.draw_site_frame(site_names=['IFtip', 'MFtip','RFtip'])

            # jac = sim.get_site_Jac('MFtip')
            # viz.viewer.user_scn.ngeom = i
            # print(viz.viewer.user_scn.ngeom)
            # Render
            viz.render()

            elapsed = time.time() - start_time
            sim_time = sim.data.time - sim_start_time
            if sim_time > elapsed:
                time.sleep(sim_time - elapsed)